# 🔬 Thesis DS Experiment Analysis - Pilot 08/12/2025

**Contents:**
1. Setup & Data Loading
2. Helper Functions
3. Overview & Summary
4. DevTools Analysis
5. Dropout Analysis
6. Time Statistics
7. Performance Analysis (Confusion Matrix)
8. User Rankings
9. Deep Dive Analysis

In [30]:
import sqlite3
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

In [31]:
# ============================================================================
# 📦 HELPER FUNCTIONS
# ============================================================================

def print_title(title):
    """Print a formatted section title"""
    print("\n" + "=" * 60)
    print(f"📊 {title}")
    print("=" * 60)

def calc_confusion_matrix(df, user_id=None):
    """Calculate confusion matrix metrics for a user or all users"""
    if user_id:
        data = df[df['user_id'] == user_id]
    else:
        data = df
    
    # True labels (correct_classification) vs Predicted (classification_decision)
    TP = len(data[(data['correct_classification'] == 'signal') & (data['classification_decision'] == 'signal')])
    TN = len(data[(data['correct_classification'] == 'noise') & (data['classification_decision'] == 'noise')])
    FP = len(data[(data['correct_classification'] == 'noise') & (data['classification_decision'] == 'signal')])
    FN = len(data[(data['correct_classification'] == 'signal') & (data['classification_decision'] == 'noise')])
    
    total = TP + TN + FP + FN
    accuracy = (TP + TN) / total if total > 0 else 0
    hit_rate = TP / (TP + FN) if (TP + FN) > 0 else 0
    fa_rate = FP / (FP + TN) if (FP + TN) > 0 else 0
    
    return {
        'TP': TP, 'TN': TN, 'FP': FP, 'FN': FN,
        'accuracy': accuracy,
        'hit_rate': hit_rate,
        'false_alarm_rate': fa_rate,
        'total_trials': total
    }

def time_stats(series):
    """Calculate time statistics"""
    return {
        'mean': series.mean(),
        'min': series.min(),
        'max': series.max(),
        'std': series.std(),
        'median': series.median()
    }

def format_time(seconds):
    """Format seconds to mm:ss"""
    mins = int(seconds // 60)
    secs = int(seconds % 60)
    return f"{mins}:{secs:02d}"

print("✅ Helper functions loaded!")


✅ Helper functions loaded!


In [32]:
# =============================================================================
# ⚠️  PLACEHOLDER CELL - Validation moved to cell 5 (after data loading)
# =============================================================================
# This cell was trying to run validation before data was loaded
# The validation code has been moved to cell 5, which runs AFTER data loading
# You can delete this cell or leave it as a placeholder

print("⚠️  Validation cell moved to cell 5 (after data loading)")
print("   Run cell 5 after loading data to see validation results")


⚠️  Validation cell moved to cell 5 (after data loading)
   Run cell 5 after loading data to see validation results


In [33]:
# =============================================================================
# STEP 1: Extract data from SQLite DB (from PythonAnywhere)
# =============================================================================
DATA_FOLDER = 'data/old_data_0912'  # Change this when you have new data

# Connect to the database
conn = sqlite3.connect(f'{DATA_FOLDER}/db.sqlite3')

# Extract ExperimentData (participants)
# Try to include csv_row_id, but handle if column doesn't exist (before migration)
try:
    users_df = pd.read_sql_query("""
        SELECT user_id, aid, csv_row_id, ps, human_sensitivity, ds_sensitivity,
               start_time, complete, end_time
        FROM experiment_experimentdata
    """, conn)
    print("✅ Loaded users with csv_row_id column")
except Exception as e:
    if 'csv_row_id' in str(e):
        # Column doesn't exist - load without it and add as None
        print("⚠️  csv_row_id column not found - loading without it (migration not run yet)")
        users_df = pd.read_sql_query("""
            SELECT user_id, aid, ps, human_sensitivity, ds_sensitivity,
                   start_time, complete, end_time
            FROM experiment_experimentdata
        """, conn)
        users_df['csv_row_id'] = None
    else:
        raise

# Extract ExperimentAction (trial decisions)
actions_df = pd.read_sql_query("""
    SELECT ea.user_id_id as user_id, ea.block_number, ea.trial_number,
           ea.classification_decision, ea.stimulus_seen, ea.dss_judgment,
           ea.decision_time, ea.correct_classification
    FROM experiment_experimentaction ea
""", conn)

# Extract TOAST (questionnaire - including numeracy and demographics)
toast_df = pd.read_sql_query("""
    SELECT tr.user_id_id as user_id, 
           tr.usefulness, tr.reliability, tr.trust, tr.confidence,
           tr.satisfaction, tr.predictability, tr.understandability, 
           tr.surprised, tr.comfortable,
           tr.numeracy_fractions, tr.numeracy_shirt, tr.numeracy_useful,
           tr.age_group, tr.gender, tr.education
    FROM experiment_toastresponse tr
""", conn)

conn.close()

# =============================================================================
# STEP 2: Load additional CSVs (not in DB)
# =============================================================================

# Conditions file (experimental design - what each user saw)
conditions_df = pd.read_csv(f'{DATA_FOLDER}/conditions_experiment_3ps_11x11_120_A.csv')

# DevTools detection log (separate CSV)
try:
    devtools_df = pd.read_csv(f'{DATA_FOLDER}/devtools_log.csv')
    print(f"DevTools log: {len(devtools_df)} entries")
except:
    devtools_df = pd.DataFrame()
    print("No DevTools log found")

# =============================================================================
# STEP 3: Save extracted data to CSV (for backup/sharing)
# =============================================================================
users_df.to_csv(f'{DATA_FOLDER}/experiment_data.csv', index=False)
actions_df.to_csv(f'{DATA_FOLDER}/experiment_actions.csv', index=False)
toast_df.to_csv(f'{DATA_FOLDER}/TOAST.csv', index=False)

print("✅ Data loaded successfully!")
print(f"Total participants: {len(users_df)}")
print(f"Completed: {users_df['complete'].sum()}")
print(f"Total actions: {len(actions_df)}")
print(f"TOAST responses: {len(toast_df)}")
print(f"Conditions rows: {len(conditions_df)}")

✅ Loaded users with csv_row_id column
DevTools log: 485 entries
✅ Data loaded successfully!
Total participants: 64
Completed: 52
Total actions: 6379
TOAST responses: 52
Conditions rows: 363


In [34]:
# =============================================================================
# UPDATE: Add csv_row_id column and fix ps/dprime values in users_df
# =============================================================================
print_title("Updating users_df with csv_row_id and correct values")

# Load mapping file
mapping_df = pd.read_csv(f'{DATA_FOLDER}/user_csv_row_mapping.csv')
print(f"✅ Loaded mapping: {len(mapping_df)} users")

# Load conditions CSV to get correct ps, dprime_h, dprime_s
conditions_file = 'data/conditions_experiment_3ps_11x11_120_A.csv'
conditions_df_full = pd.read_csv(conditions_file)
print(f"✅ Loaded conditions CSV: {len(conditions_df_full)} rows")

# Add csv_row_id column to users_df (initialize as None)
if 'csv_row_id' not in users_df.columns:
    users_df['csv_row_id'] = None
    print("✅ Added csv_row_id column to users_df")

# Update users_df with csv_row_id and correct ps/dprime values from mapping
updated = 0
not_found = []

for _, row in mapping_df.iterrows():
    user_id = int(row['user_id'])  # Already int now
    csv_row_id = int(row['csv_row_id'])
    
    # Find user in users_df
    user_mask = users_df['user_id'] == user_id
    if user_mask.any():
        # Get correct ps, dprime_h, dprime_s from CSV row
        csv_row = conditions_df_full[conditions_df_full['id'] == csv_row_id].iloc[0]
        
        # Update users_df
        users_df.loc[user_mask, 'csv_row_id'] = csv_row_id
        users_df.loc[user_mask, 'ps'] = float(csv_row['ps'])
        users_df.loc[user_mask, 'human_sensitivity'] = float(csv_row['dprime_h'])
        users_df.loc[user_mask, 'ds_sensitivity'] = float(csv_row['dprime_s'])
        
        updated += 1
    else:
        not_found.append(user_id)

# Summary
print(f"\n✅ Updated {updated} users in users_df")
if not_found:
    print(f"⚠️  Not found in users_df: {len(not_found)} users {not_found[:10]}")

# Show sample
print(f"\n📊 Sample updated users:")
print(users_df[['user_id', 'csv_row_id', 'ps', 'human_sensitivity', 'ds_sensitivity']].head(10))

print("\n" + "=" * 60)
print("✅ users_df updated!")
print("   - csv_row_id column added")
print("   - ps, dprime_h, dprime_s updated from correct CSV rows")
# Update conditions_df - set isDemo=1 for all used rows (old users)
conditions_df.loc[conditions_df['used'] == 1, 'isDemo'] = 1
print(f"✅ Set isDemo=1 for {conditions_df['used'].sum()} used rows in conditions_df")
print("=" * 60)



📊 Updating users_df with csv_row_id and correct values
✅ Loaded mapping: 52 users
✅ Loaded conditions CSV: 363 rows

✅ Updated 52 users in users_df

📊 Sample updated users:
   user_id  csv_row_id    ps  human_sensitivity  ds_sensitivity
0       31       237.0  0.35                2.5             1.5
1       32       276.0  0.50                1.1             0.5
2       33       236.0  0.35                2.5             1.3
3       34       257.0  0.50                0.7             1.1
4       35        74.0  0.20                1.7             1.9
5       36       218.0  0.35                2.1             2.1
6       37        89.0  0.20                2.1             0.5
7       38       170.0  0.35                1.3             1.3
8       39         NaN  0.35                1.7             0.7
9       40         NaN  0.50                2.3             1.7

✅ users_df updated!
   - csv_row_id column added
   - ps, dprime_h, dprime_s updated from correct CSV rows
✅ Set isDemo=1

In [35]:
# =============================================================================
# ✅ VALIDATION CHECKS: CSV Row Assignment & Data Integrity
# =============================================================================
print_title("Validation: CSV Row Assignment & Data Integrity")

# Check if csv_row_id column exists in users_df (from database)
if 'csv_row_id' in users_df.columns:
    print("✅ csv_row_id column exists in database")
else:
    print("⚠️  WARNING: csv_row_id column NOT found in database")
    print("   This means the migration hasn't been run yet, or data is from before the fix")
    users_df['csv_row_id'] = None

# =============================================================================
# CHECK 1: Each user should have exactly 1 csv_row_id
# =============================================================================
print("\n" + "-" * 60)
print("CHECK 1: Each user has exactly 1 csv_row_id")
print("-" * 60)

users_with_row = users_df[users_df['csv_row_id'].notna()]
users_without_row = users_df[users_df['csv_row_id'].isna()]

print(f"Users with csv_row_id: {len(users_with_row)}")
print(f"Users without csv_row_id: {len(users_without_row)}")

if len(users_without_row) > 0:
    print(f"\n⚠️  Users without csv_row_id: {list(users_without_row['user_id'].values)}")
    print("   These are likely from before the fix was implemented")

# Check for duplicate csv_row_id assignments (should be 0 for completed users)
completed_users = users_df[users_df['complete'] == True]
if len(completed_users) > 0 and 'csv_row_id' in completed_users.columns:
    row_counts = completed_users['csv_row_id'].value_counts()
    duplicate_rows = row_counts[row_counts > 1]
    
    if len(duplicate_rows) > 0:
        print(f"\n❌ ERROR: {len(duplicate_rows)} CSV rows assigned to multiple completed users:")
        for row_id, count in duplicate_rows.items():
            user_ids = completed_users[completed_users['csv_row_id'] == row_id]['user_id'].tolist()
            print(f"   Row {row_id}: {count} users {user_ids}")
    else:
        print("✅ Each completed user has unique csv_row_id")

# =============================================================================
# CHECK 2: CSV rows with used=0 should NOT be assigned to any user
# =============================================================================
print("\n" + "-" * 60)
print("CHECK 2: CSV rows with used=0 are not assigned to users")
print("-" * 60)

unused_rows = conditions_df[conditions_df['used'] == 0]['id'].tolist()
if len(users_with_row) > 0:
    assigned_rows = users_with_row['csv_row_id'].unique().tolist()
    conflict_rows = [r for r in unused_rows if r in assigned_rows]
    
    if len(conflict_rows) > 0:
        print(f"❌ ERROR: {len(conflict_rows)} unused CSV rows are assigned to users:")
        for row_id in conflict_rows[:10]:  # Show first 10
            user_ids = users_with_row[users_with_row['csv_row_id'] == row_id]['user_id'].tolist()
            print(f"   Row {row_id}: users {user_ids}")
        if len(conflict_rows) > 10:
            print(f"   ... and {len(conflict_rows) - 10} more")
    else:
        print(f"✅ All {len(unused_rows)} unused CSV rows are not assigned to any user")
else:
    print("⚠️  Skipping: No users with csv_row_id to check")

# =============================================================================
# CHECK 3: CSV rows with used=1 should be assigned to exactly 1 user
# =============================================================================
print("\n" + "-" * 60)
print("CHECK 3: CSV rows with used=1 are assigned to exactly 1 user")
print("-" * 60)

used_rows = conditions_df[conditions_df['used'] == 1]['id'].tolist()
if len(users_with_row) > 0:
    # Count how many users are assigned to each used row
    used_row_assignments = users_with_row[users_with_row['csv_row_id'].isin(used_rows)]['csv_row_id'].value_counts()
    
    # Check for rows with 0 or >1 assignments
    rows_with_zero = [r for r in used_rows if r not in used_row_assignments.index]
    rows_with_many = used_row_assignments[used_row_assignments > 1]
    
    if len(rows_with_zero) > 0:
        print(f"⚠️  WARNING: {len(rows_with_zero)} used CSV rows are not assigned to any user:")
        print(f"   Rows: {rows_with_zero[:10]}")
        if len(rows_with_zero) > 10:
            print(f"   ... and {len(rows_with_zero) - 10} more")
    
    if len(rows_with_many) > 0:
        print(f"❌ ERROR: {len(rows_with_many)} used CSV rows are assigned to multiple users:")
        for row_id, count in rows_with_many.items():
            user_ids = users_with_row[users_with_row['csv_row_id'] == row_id]['user_id'].tolist()
            print(f"   Row {row_id}: {count} users {user_ids}")
    
    rows_with_one = used_row_assignments[used_row_assignments == 1]
    if len(rows_with_one) == len(used_rows) - len(rows_with_zero) - len(rows_with_many):
        print(f"✅ {len(rows_with_one)} used CSV rows are correctly assigned to exactly 1 user")
else:
    print("⚠️  Skipping: No users with csv_row_id to check")

# =============================================================================
# CHECK 4: Sample verification - Compare user trials to their CSV row
# =============================================================================
print("\n" + "-" * 60)
print("CHECK 4: Sample verification - User trials vs CSV row")
print("-" * 60)

if len(users_with_row) > 0:
    # Select a few sample users to verify
    sample_users = users_with_row.head(3)['user_id'].tolist()
    
    STIMULI_SCALAR = 6.5  # From views.py
    
    for user_id in sample_users:
        user_row_id = users_with_row[users_with_row['user_id'] == user_id]['csv_row_id'].iloc[0]
        csv_row = conditions_df[conditions_df['id'] == user_row_id].iloc[0]
        
        # Get user's actions
        user_actions = actions_df[actions_df['user_id'] == user_id].sort_values(['block_number', 'trial_number'])
        
        print(f"\n📋 User {user_id} (CSV row {user_row_id}):")
        
        # Check Block 1 (trials 1-10, CSV columns 1-10)
        block1_actions = user_actions[user_actions['block_number'] == 1]
        if len(block1_actions) > 0:
            mismatches = 0
            for idx, action in block1_actions.iterrows():
                trial_num = int(action['trial_number'])
                csv_col = f'0{trial_num}' if trial_num < 10 else f'{trial_num}'
                
                # Compare event type
                csv_event = csv_row[f'event_t{csv_col}']
                user_event = action['correct_classification']
                if csv_event != user_event:
                    mismatches += 1
            
            if mismatches == 0:
                print(f"   ✅ Block 1: All {len(block1_actions)} trials match CSV row")
            else:
                print(f"   ❌ Block 1: {mismatches}/{len(block1_actions)} trials don't match CSV row")
        
        # Check Block 2 (trials 11-20, CSV columns 11-20)
        block2_actions = user_actions[user_actions['block_number'] == 2]
        if len(block2_actions) > 0:
            mismatches = 0
            for idx, action in block2_actions.iterrows():
                trial_num = int(action['trial_number'])
                csv_trial = trial_num + 10  # Block 2 trial 1 = CSV column 11
                csv_col = f'0{csv_trial}' if csv_trial < 10 else f'{csv_trial}'
                
                csv_event = csv_row[f'event_t{csv_col}']
                user_event = action['correct_classification']
                if csv_event != user_event:
                    mismatches += 1
            
            if mismatches == 0:
                print(f"   ✅ Block 2: All {len(block2_actions)} trials match CSV row")
            else:
                print(f"   ❌ Block 2: {mismatches}/{len(block2_actions)} trials don't match CSV row")
        
        # Check Block 3 (trials 1-100, CSV columns 21-120) - IMPORTANT!
        block3_actions = user_actions[user_actions['block_number'] == 3]
        if len(block3_actions) > 0:
            mismatches = 0
            for idx, action in block3_actions.iterrows():
                trial_num = int(action['trial_number'])
                csv_trial = trial_num + 20  # Block 3 trial 1 = CSV column 21 (FIXED!)
                csv_col = f'0{csv_trial}' if csv_trial < 10 else f'{csv_trial}'
                
                csv_event = csv_row[f'event_t{csv_col}']
                user_event = action['correct_classification']
                if csv_event != user_event:
                    mismatches += 1
            
            if mismatches == 0:
                print(f"   ✅ Block 3: All {len(block3_actions)} trials match CSV row (columns 21-120)")
            else:
                print(f"   ❌ Block 3: {mismatches}/{len(block3_actions)} trials don't match CSV row")
                print(f"      ⚠️  This might indicate Block 3 is using wrong columns (1-100 instead of 21-120)")
else:
    print("⚠️  Skipping: No users with csv_row_id to verify")

# =============================================================================
# CHECK 5: Verify Block 3 column mapping (21-120, not 1-100)
# =============================================================================
print("\n" + "-" * 60)
print("CHECK 5: Block 3 column mapping verification")
print("-" * 60)

if len(users_with_row) > 0:
    # Check if Block 3 trial 1 matches CSV column 21 (not column 1)
    sample_user = users_with_row.iloc[0]
    user_id = sample_user['user_id']
    csv_row_id = sample_user['csv_row_id']
    
    csv_row = conditions_df[conditions_df['id'] == csv_row_id].iloc[0]
    user_actions = actions_df[actions_df['user_id'] == user_id]
    block3_trial1 = user_actions[(user_actions['block_number'] == 3) & (user_actions['trial_number'] == 1)]
    
    if len(block3_trial1) > 0:
        trial1_event = block3_trial1.iloc[0]['correct_classification']
        csv_col1_event = csv_row['event_t01']  # Column 1
        csv_col21_event = csv_row['event_t21']  # Column 21 (correct)
        
        if trial1_event == csv_col21_event:
            print(f"✅ Block 3 trial 1 matches CSV column 21 (CORRECT mapping)")
        elif trial1_event == csv_col1_event:
            print(f"❌ ERROR: Block 3 trial 1 matches CSV column 1 (WRONG - should be 21)")
            print(f"   This indicates Block 3 is using columns 1-100 instead of 21-120")
        else:
            print(f"⚠️  Block 3 trial 1 doesn't match column 1 or 21 - needs investigation")
    else:
        print("⚠️  Could not find Block 3 trial 1 for verification")
else:
    print("⚠️  Skipping: No users with csv_row_id to verify")

# =============================================================================
# SUMMARY
# =============================================================================
print("\n" + "=" * 60)
print("📊 VALIDATION SUMMARY")
print("=" * 60)

issues_found = []
if len(users_without_row) > 0:
    issues_found.append(f"{len(users_without_row)} users without csv_row_id")
if len(users_with_row) > 0:
    if 'csv_row_id' in completed_users.columns:
        duplicate_rows = completed_users['csv_row_id'].value_counts()
        if len(duplicate_rows[duplicate_rows > 1]) > 0:
            issues_found.append("Duplicate csv_row_id assignments")

if len(issues_found) == 0:
    print("✅ All validation checks passed!")
else:
    print(f"⚠️  Found {len(issues_found)} potential issues:")
    for issue in issues_found:
        print(f"   - {issue}")

print("\n" + "=" * 60)



📊 Validation: CSV Row Assignment & Data Integrity
✅ csv_row_id column exists in database

------------------------------------------------------------
CHECK 1: Each user has exactly 1 csv_row_id
------------------------------------------------------------
Users with csv_row_id: 52
Users without csv_row_id: 12

⚠️  Users without csv_row_id: [np.int64(39), np.int64(40), np.int64(41), np.int64(43), np.int64(58), np.int64(60), np.int64(64), np.int64(68), np.int64(75), np.int64(80), np.int64(87), np.int64(93)]
   These are likely from before the fix was implemented

❌ ERROR: 1 CSV rows assigned to multiple completed users:
   Row 305.0: 2 users [56, 76]

------------------------------------------------------------
CHECK 2: CSV rows with used=0 are not assigned to users
------------------------------------------------------------
✅ All 303 unused CSV rows are not assigned to any user

------------------------------------------------------------
CHECK 3: CSV rows with used=1 are assigned to 

In [36]:
# =============================================================================
# VERIFICATION: Compare 2 users' data to their CSV rows
# =============================================================================
print_title("Verification: User Data vs CSV Row")

# Select 2 users from mapping
sample_users = [31, 32]  # Change these to any user IDs you want to check

for user_id in sample_users:
    print(f"\n{'='*60}")
    print(f"USER {user_id}")
    print(f"{'='*60}")
    
    # Get user's csv_row_id from mapping
    user_mapping = mapping_df[mapping_df['user_id'] == float(user_id)]
    if len(user_mapping) == 0:
        print(f"⚠️  User {user_id} not found in mapping")
        continue
    
    csv_row_id = int(user_mapping['csv_row_id'].iloc[0])
    print(f"\n📋 CSV Row ID: {csv_row_id}")
    
    # Get CSV row
    csv_row = conditions_df_full[conditions_df_full['id'] == csv_row_id].iloc[0]
    
    # Show first 30 columns of CSV row
    print(f"\n📊 CSV Row - First 30 columns:")
    print(f"   id: {csv_row['id']}")
    print(f"   ps: {csv_row['ps']}")
    print(f"   dprime_h: {csv_row['dprime_h']}")
    print(f"   dprime_s: {csv_row['dprime_s']}")
    print(f"\n   Event columns (first 10):")
    for i in range(1, 11):
        col = f'event_t{str(i).zfill(2)}'
        print(f"      {col}: {csv_row[col]}")
    
    # Get user's actions
    user_actions = actions_df[actions_df['user_id'] == user_id].sort_values(['block_number', 'trial_number'])
    
    print(f"\n📝 User {user_id} Actions:")
    print(f"   Total actions: {len(user_actions)}")
    print(f"   Blocks: {sorted(user_actions['block_number'].unique())}")
    
    # Compare Block 1 (trials 1-10, CSV columns 1-10)
    print(f"\n   🔍 Block 1 Comparison (trials 1-10 vs CSV columns 1-10):")
    block1 = user_actions[user_actions['block_number'] == 1].sort_values('trial_number')
    matches = 0
    mismatches = []
    for idx, action in block1.iterrows():
        trial_num = int(action['trial_number'])
        csv_col = f'event_t{str(trial_num).zfill(2)}'
        csv_event = csv_row[csv_col]
        user_event = action['correct_classification']
        
        if csv_event == user_event:
            matches += 1
        else:
            mismatches.append(f"Trial {trial_num}: CSV={csv_event}, User={user_event}")
    
    print(f"      ✅ Matches: {matches}/{len(block1)}")
    if mismatches:
        print(f"      ❌ Mismatches:")
        for mm in mismatches[:5]:
            print(f"         {mm}")
    
    # Compare Block 2 (trials 11-20, CSV columns 11-20)
    print(f"\n   🔍 Block 2 Comparison (trials 11-20 vs CSV columns 11-20):")
    block2 = user_actions[user_actions['block_number'] == 2].sort_values('trial_number')
    matches = 0
    mismatches = []
    for idx, action in block2.iterrows():
        trial_num = int(action['trial_number'])
        csv_trial = trial_num + 10  # Block 2 trial 1 = CSV column 11
        csv_col = f'event_t{str(csv_trial).zfill(2)}'
        csv_event = csv_row[csv_col]
        user_event = action['correct_classification']
        
        if csv_event == user_event:
            matches += 1
        else:
            mismatches.append(f"Trial {trial_num}: CSV={csv_event}, User={user_event}")
    
    print(f"      ✅ Matches: {matches}/{len(block2)}")
    if mismatches:
        print(f"      ❌ Mismatches:")
        for mm in mismatches[:5]:
            print(f"         {mm}")
    
    # Compare Block 3 (trials 1-100, CSV columns 21-120) - IMPORTANT!
    print(f"\n   🔍 Block 3 Comparison (trials 1-100 vs CSV columns 21-120):")
    print(f"      Note: Block 3 trials 1-20 are repeats (old bug), so we check trials 21-100")
    block3 = user_actions[user_actions['block_number'] == 3].sort_values('trial_number')
    
    # Check trials 21-100 (should match CSV columns 41-120)
    block3_unique = block3[block3['trial_number'] >= 21]
    matches = 0
    mismatches = []
    for idx, action in block3_unique.iterrows():
        trial_num = int(action['trial_number'])
        csv_trial = trial_num + 20  # Block 3 trial 21 = CSV column 41
        csv_col = f'event_t{str(csv_trial).zfill(2)}'
        csv_event = csv_row[csv_col]
        user_event = action['correct_classification']
        
        if csv_event == user_event:
            matches += 1
        else:
            mismatches.append(f"Trial {trial_num}: CSV={csv_event}, User={user_event}")
    
    print(f"      ✅ Matches (trials 21-100): {matches}/{len(block3_unique)}")
    if mismatches:
        print(f"      ❌ Mismatches:")
        for mm in mismatches[:5]:
            print(f"         {mm}")
    
    # Show sample actions
    print(f"\n   📋 Sample Actions (first 5):")
    for idx, action in user_actions.head(5).iterrows():
        print(f"      Block {action['block_number']}, Trial {action['trial_number']}: "
              f"{action['correct_classification']} (decision: {action['classification_decision']})")

print("\n" + "="*60)
print("✅ Verification complete!")
print("="*60)



📊 Verification: User Data vs CSV Row

USER 31

📋 CSV Row ID: 237

📊 CSV Row - First 30 columns:
   id: 237
   ps: 0.35
   dprime_h: 2.5
   dprime_s: 1.5

   Event columns (first 10):
      event_t01: noise
      event_t02: signal
      event_t03: noise
      event_t04: signal
      event_t05: noise
      event_t06: noise
      event_t07: signal
      event_t08: noise
      event_t09: signal
      event_t10: signal

📝 User 31 Actions:
   Total actions: 120
   Blocks: [np.int64(1), np.int64(2), np.int64(3)]

   🔍 Block 1 Comparison (trials 1-10 vs CSV columns 1-10):
      ✅ Matches: 10/10

   🔍 Block 2 Comparison (trials 11-20 vs CSV columns 11-20):
      ✅ Matches: 10/10

   🔍 Block 3 Comparison (trials 1-100 vs CSV columns 21-120):
      Note: Block 3 trials 1-20 are repeats (old bug), so we check trials 21-100
      ✅ Matches (trials 21-100): 39/80
      ❌ Mismatches:
         Trial 22: CSV=noise, User=signal
         Trial 24: CSV=noise, User=signal
         Trial 28: CSV=signal, Us

In [37]:
# ============================================================================
# 🔧 DATA FILTERING - Updated DevTools Policy
# ============================================================================
print_title("Data Filtering - Updated DevTools Policy")

# Parse DevTools log to identify suspicious users
import json
if len(devtools_df) > 0:
    devtools_df['parsed'] = devtools_df['details'].apply(lambda x: json.loads(x) if isinstance(x, str) else {})
    devtools_df['trial'] = devtools_df['parsed'].apply(lambda x: x.get('trial', 0))
    devtools_df['method'] = devtools_df['parsed'].apply(lambda x: x.get('method', 'unknown'))
    
    # Count events per user
    devtools_counts = devtools_df.groupby('user_id').size().reset_index(name='count')
    
    # Identify suspicious DevTools users (high performance + many events)
    # Only exclude if: many events (>50) AND suspiciously high performance (>90th percentile)
    block3 = actions_df[actions_df['block_number'] == 3]
    user_perf = []
    for uid in block3['user_id'].unique():
        user_data = block3[block3['user_id'] == uid]
        acc = (user_data['classification_decision'] == user_data['correct_classification']).mean()
        user_perf.append({'user_id': uid, 'accuracy': acc})
    perf_df = pd.DataFrame(user_perf)
    
    # Calculate 90th percentile threshold
    perf_90th = perf_df['accuracy'].quantile(0.90)
    
    # Find suspicious users
    suspicious_users = []
    for _, row in devtools_counts.iterrows():
        user_id = row['user_id']
        count = row['count']
        user_acc = perf_df[perf_df['user_id'] == user_id]['accuracy'].iloc[0] if len(perf_df[perf_df['user_id'] == user_id]) > 0 else 0
        
        # Only exclude if many events AND high performance
        if count > 50 and user_acc > perf_90th:
            suspicious_users.append(user_id)
    
    print(f"DevTools users detected: {len(devtools_counts)}")
    print(f"Suspicious users (excluded): {len(suspicious_users)}")
    if suspicious_users:
        print(f"  User IDs: {suspicious_users}")
    print(f"\n✅ Updated policy: Only exclude DevTools users with:")
    print(f"   - Many events (>50) AND")
    print(f"   - Suspiciously high performance (>90th percentile)")
else:
    suspicious_users = []
    print("No DevTools detections found")

# Filter data - exclude only suspicious DevTools users and incomplete users
incomplete_user_ids = users_df[users_df['complete'] == False]['user_id'].tolist()
excluded_user_ids = list(set(incomplete_user_ids + suspicious_users))

users_clean = users_df[~users_df['user_id'].isin(excluded_user_ids)]
actions_clean = actions_df[~actions_df['user_id'].isin(excluded_user_ids)]
toast_clean = toast_df[~toast_df['user_id'].isin(excluded_user_ids)]

print(f"\n📊 Filtered Data:")
print(f"   Total users: {len(users_df)}")
print(f"   Excluded: {len(excluded_user_ids)} (incomplete: {len(incomplete_user_ids)}, suspicious DevTools: {len(suspicious_users)})")
print(f"   Clean users for analysis: {len(users_clean)}")
print(f"   Clean actions: {len(actions_clean)}")
print(f"   Clean TOAST responses: {len(toast_clean)}")




📊 Data Filtering - Updated DevTools Policy
DevTools users detected: 8
Suspicious users (excluded): 0

✅ Updated policy: Only exclude DevTools users with:
   - Many events (>50) AND
   - Suspiciously high performance (>90th percentile)

📊 Filtered Data:
   Total users: 64
   Excluded: 12 (incomplete: 12, suspicious DevTools: 0)
   Clean users for analysis: 52
   Clean actions: 6240
   Clean TOAST responses: 52


In [38]:
# ============================================================================
# 📋 QUICK OVERVIEW
# ============================================================================
print_title("Quick Overview")

print(f"""
👥 PARTICIPANTS: {len(users_df)} total | {users_df['complete'].sum()} completed | {len(users_df) - users_df['complete'].sum()} dropped
📊 COMPLETION RATE: {users_df['complete'].mean()*100:.1f}%

🎮 ACTIONS: {len(actions_df)} total | {actions_df['user_id'].nunique()} users with actions
📝 TOAST RESPONSES: {len(toast_df)}
🔧 DEVTOOLS DETECTIONS: {len(devtools_df)} ({devtools_df['user_id'].nunique() if len(devtools_df) > 0 else 0} unique users)

📊 DEMOGRAPHICS:
   Age: {toast_df['age_group'].value_counts().to_dict()}
   Gender: {toast_df['gender'].value_counts().to_dict()}
""")


📊 Quick Overview

👥 PARTICIPANTS: 64 total | 52 completed | 12 dropped
📊 COMPLETION RATE: 81.2%

🎮 ACTIONS: 6379 total | 59 users with actions
📝 TOAST RESPONSES: 52
🔧 DEVTOOLS DETECTIONS: 485 (8 unique users)

📊 DEMOGRAPHICS:
   Age: {'41-55': 16, '56-70': 13, 'older_than_70': 10, '35-40': 7, '26-34': 5, '18-25': 1}
   Gender: {'female': 31, 'male': 21}



In [39]:
# =============================================================================
# DETAILED DATA VIEWS
# =============================================================================

# Show all dataframes
print("=" * 60)
print("USERS DataFrame")
print("=" * 60)
display(users_df)

print("\n" + "=" * 60)
print("TOAST DataFrame (first 10 rows)")
print("=" * 60)
display(toast_df.head(10))

print("\n" + "=" * 60)
print("ACTIONS DataFrame (sample)")
print("=" * 60)
display(actions_df.sample(min(10, len(actions_df))))

print("\n" + "=" * 60)
print("DEVTOOLS LOG")
print("=" * 60)
display(devtools_df)



USERS DataFrame


,user_id,aid,csv_row_id,ps,human_sensitivity,ds_sensitivity,start_time,complete,end_time
0,31,6936c6d0-7fa9-685b-1e07-ae6e84bef2a3,237.0,0.35,2.5,1.5,2025-12-08 12:40:00.683731,1,2025-12-08 12:49:37.435722
1,32,6936c71b-a0a3-ee37-16eb-65cc8a66701b,276.0,0.50,1.1,0.5,2025-12-08 12:41:04.089556,1,2025-12-08 12:51:04.437791
2,33,6936c751-1136-ec6f-6e43-fb4c06483451,236.0,0.35,2.5,1.3,2025-12-08 12:41:38.736285,1,2025-12-08 12:52:57.560370
3,34,6936c75c-7fa1-868f-1f22-38e824c1761b,257.0,0.50,0.7,1.1,2025-12-08 12:42:10.259373,1,2025-12-08 12:47:35.700894
4,35,6936c73a-b5a9-20ce-0086-3a8e34062239,74.0,0.20,1.7,1.9,2025-12-08 12:42:21.318183,1,2025-12-08 13:05:20.150141
...,...,...,...,...,...,...,...,...,...
59,90,69372350-20fe-fc50-864a-e1a4b67f65be,79.0,0.20,1.9,0.7,2025-12-08 19:15:08.131191,1,2025-12-08 19:25:19.753789
60,91,6937236f-8b0f-538c-bffb-f79f2724dddd,184.0,0.35,1.5,1.9,2025-12-08 19:15:09.316309,1,2025-12-08 19:26:28.311498
61,92,6937236c-bded-6edc-73d8-02a325e45fb2,95.0,0.20,2.1,1.7,2025-12-08 19:15:24.560104,1,2025-12-08 19:32:38.552176
62,93,6937236e-5ab1-810d-9f7c-0d4055bf78a2,NaN,0.20,1.1,0.5,2025-12-08 19:15:43.407867,0,None



TOAST DataFrame (first 10 rows)


,user_id,usefulness,reliability,trust,confidence,satisfaction,predictability,understandability,surprised,comfortable,numeracy_fractions,numeracy_shirt,numeracy_useful,age_group,gender,education
0,34,6,5,6,6,4,5,4,6,5,5,6,5,41-55,female,some_high_school
1,36,7,6,7,7,5,5,6,6,5,5,5,5,41-55,female,bachelor
2,31,6,6,2,2,2,5,4,3,4,3,3,4,41-55,female,bachelor
3,38,4,5,5,5,5,5,5,5,5,6,6,5,older_than_70,female,bachelor
4,37,3,3,3,3,3,3,3,3,3,2,2,2,26-34,male,high_school
5,32,7,6,6,6,7,5,6,5,5,3,4,6,56-70,male,trade_technical
6,42,2,2,2,2,1,1,1,1,1,2,2,2,35-40,female,master
7,44,5,5,7,7,4,5,5,5,3,4,4,4,41-55,female,some_high_school
8,33,4,5,6,7,6,6,6,6,6,5,5,5,35-40,female,some_college
9,49,4,5,6,6,5,5,6,5,5,5,5,5,older_than_70,male,bachelor



ACTIONS DataFrame (sample)


,user_id,block_number,trial_number,classification_decision,stimulus_seen,dss_judgment,decision_time,correct_classification
710,37,3,37,signal,7.96,noise,1.611660,signal
3973,59,3,81,noise,7.10,noise,3.960742,signal
4890,73,3,21,noise,6.95,noise,4.116880,signal
827,42,3,30,signal,6.61,signal,1.421999,signal
4563,72,3,23,noise,3.26,noise,2.428234,noise
5480,85,3,81,signal,6.91,signal,2.982607,signal
2971,67,3,42,signal,7.85,signal,2.292168,signal
1879,35,3,55,noise,6.16,noise,1.803239,noise
4170,81,2,3,signal,6.71,signal,2.231210,signal
675,38,3,50,noise,5.69,noise,1.633234,noise



DEVTOOLS LOG


,user_id,details,timestamp,parsed,trial,method
0,22,"{""trial"":0,""method"":""window_size""}",2025-12-04T20:37:19.194993,"{'trial': 0, 'method': 'window_size'}",0,window_size
1,24,"{""trial"":0,""method"":""window_size""}",2025-12-04T20:37:56.957378,"{'trial': 0, 'method': 'window_size'}",0,window_size
2,49,"{""trial"":0,""method"":""window_size""}",2025-12-08T12:49:43.097766,"{'trial': 0, 'method': 'window_size'}",0,window_size
3,49,"{""trial"":0,""method"":""window_size""}",2025-12-08T12:51:39.716617,"{'trial': 0, 'method': 'window_size'}",0,window_size
4,67,"{""trial"":0,""method"":""window_size""}",2025-12-08T18:14:13.260505,"{'trial': 0, 'method': 'window_size'}",0,window_size
...,...,...,...,...,...,...
480,91,"{""trial"":0,""method"":""window_size""}",2025-12-08T19:24:42.895483,"{'trial': 0, 'method': 'window_size'}",0,window_size
481,91,"{""trial"":0,""method"":""window_size""}",2025-12-08T19:24:44.751393,"{'trial': 0, 'method': 'window_size'}",0,window_size
482,91,"{""trial"":0,""method"":""window_size""}",2025-12-08T19:24:46.503595,"{'trial': 0, 'method': 'window_size'}",0,window_size
483,91,"{""trial"":0,""method"":""window_size""}",2025-12-08T19:24:48.684038,"{'trial': 0, 'method': 'window_size'}",0,window_size


In [40]:
# ============================================================================
# 🔍 DEVTOOLS ANALYSIS
# ============================================================================
print("\n" + "=" * 60)
print("DEVTOOLS DETECTION ANALYSIS")
print("=" * 60)

if len(devtools_df) > 0:
    # Parse the details JSON
    import json
    devtools_df['parsed'] = devtools_df['details'].apply(lambda x: json.loads(x) if isinstance(x, str) else {})
    devtools_df['trial'] = devtools_df['parsed'].apply(lambda x: x.get('trial', 'N/A'))
    devtools_df['method'] = devtools_df['parsed'].apply(lambda x: x.get('method', 'N/A'))

    print(f"Total DevTools detections: {len(devtools_df)}")
    print(f"Unique users: {devtools_df['user_id'].nunique()}")

    print("\n📋 DevTools Events by User:")
    for user_id in devtools_df['user_id'].unique():
        user_events = devtools_df[devtools_df['user_id'] == user_id]
        user_info = users_df[users_df['user_id'] == user_id]
        completed = user_info['complete'].values[0] if len(user_info) > 0 else 'N/A'

        print(f"\n  User {user_id} (completed: {completed}):")
        for _, row in user_events.iterrows():
            print(f"    - Trial {row['trial']}, Method: {row['method']}, Time: {row['timestamp']}")
else:
    print("No DevTools detections in this dataset")



DEVTOOLS DETECTION ANALYSIS
Total DevTools detections: 485
Unique users: 8

📋 DevTools Events by User:

  User 22 (completed: N/A):
    - Trial 0, Method: window_size, Time: 2025-12-04T20:37:19.194993

  User 24 (completed: N/A):
    - Trial 0, Method: window_size, Time: 2025-12-04T20:37:56.957378

  User 49 (completed: 1):
    - Trial 0, Method: window_size, Time: 2025-12-08T12:49:43.097766
    - Trial 0, Method: window_size, Time: 2025-12-08T12:51:39.716617

  User 67 (completed: 1):
    - Trial 0, Method: window_size, Time: 2025-12-08T18:14:13.260505
    - Trial 0, Method: window_size, Time: 2025-12-08T18:14:19.696423
    - Trial 0, Method: window_size, Time: 2025-12-08T18:14:24.431360
    - Trial 0, Method: window_size, Time: 2025-12-08T18:14:27.347808
    - Trial 0, Method: window_size, Time: 2025-12-08T18:14:29.752801
    - Trial 0, Method: window_size, Time: 2025-12-08T18:14:31.961420
    - Trial 0, Method: window_size, Time: 2025-12-08T18:14:35.396538
    - Trial 0, Method: wi

In [41]:
# ============================================================================
# 🚪 DROPOUT ANALYSIS
# ============================================================================
print_title("Dropout Analysis - Where Users Quit")

incomplete_users = users_df[users_df['complete'] == False]
print(f"Total incomplete users: {len(incomplete_users)}")

print("\n📋 Dropout Details:")
for _, user in incomplete_users.iterrows():
    user_id = user['user_id']
    user_actions = actions_df[actions_df['user_id'] == user_id]
    has_toast = len(toast_df[toast_df['user_id'] == user_id]) > 0
    
    # Calculate time spent
    start_time = pd.to_datetime(user['start_time'])
    end_time = pd.to_datetime(user['end_time']) if pd.notna(user['end_time']) else None
    time_spent = (end_time - start_time).total_seconds() if end_time else None
    
    if len(user_actions) == 0:
        dropout_point = "❌ Never started (dropped at instructions)"
    else:
        last_action = user_actions.sort_values(['block_number', 'trial_number']).iloc[-1]
        total_actions = len(user_actions)
        
        if total_actions >= 120 and has_toast:
            dropout_point = f"✅ Completed all (120 trials + TOAST) but not marked complete"
        elif total_actions >= 120:
            dropout_point = f"📋 Completed trials, dropped at TOAST questionnaire"
        else:
            dropout_point = f"🎮 Block {last_action['block_number']}, Trial {last_action['trial_number']} ({total_actions}/120 trials)"
    
    time_str = format_time(time_spent) if time_spent else "N/A"
    print(f"\n  User {user_id}:")
    print(f"    Dropout point: {dropout_point}")
    print(f"    Time spent: {time_str}")
    print(f"    Actions recorded: {len(user_actions)}")

# Summary
print("\n" + "-" * 40)
print("📊 Dropout Summary:")
no_actions = len(incomplete_users[~incomplete_users['user_id'].isin(actions_df['user_id'].unique())])
with_actions = len(incomplete_users) - no_actions
print(f"  - Dropped at instructions: {no_actions}")
print(f"  - Dropped during experiment: {with_actions}")



📊 Dropout Analysis - Where Users Quit
Total incomplete users: 12

📋 Dropout Details:

  User 39:
    Dropout point: ❌ Never started (dropped at instructions)
    Time spent: N/A
    Actions recorded: 0

  User 40:
    Dropout point: ❌ Never started (dropped at instructions)
    Time spent: N/A
    Actions recorded: 0

  User 41:
    Dropout point: ❌ Never started (dropped at instructions)
    Time spent: N/A
    Actions recorded: 0

  User 43:
    Dropout point: 🎮 Block 1, Trial 5 (5/120 trials)
    Time spent: N/A
    Actions recorded: 5

  User 58:
    Dropout point: 🎮 Block 1, Trial 2 (2/120 trials)
    Time spent: N/A
    Actions recorded: 2

  User 60:
    Dropout point: 🎮 Block 3, Trial 15 (35/120 trials)
    Time spent: N/A
    Actions recorded: 35

  User 64:
    Dropout point: ❌ Never started (dropped at instructions)
    Time spent: N/A
    Actions recorded: 0

  User 68:
    Dropout point: 🎮 Block 2, Trial 10 (20/120 trials)
    Time spent: N/A
    Actions recorded: 20

  U

In [ ]:
# ============================================================================
# 🎯 STATISTICAL ANALYSIS: Confusion Matrix by Experimental Conditions
# ============================================================================
print_title("Statistical Analysis: Performance vs Experimental Conditions")

from scipy import stats
from scipy.stats import f_oneway

# Merge actions with user conditions
block3_clean = actions_clean[actions_clean['block_number'] == 3].copy()
block3_clean = block3_clean.merge(users_clean[['user_id', 'ps', 'human_sensitivity', 'ds_sensitivity']], 
                                   on='user_id', how='left')

# Calculate user-level metrics (for statistical testing)
user_metrics = []
for user_id in block3_clean['user_id'].unique():
    user_data = block3_clean[block3_clean['user_id'] == user_id]
    user_info = users_clean[users_clean['user_id'] == user_id].iloc[0]
    
    cm = calc_confusion_matrix(user_data)
    user_metrics.append({
        'user_id': user_id,
        'ps': user_info['ps'],
        'dprime_h': user_info['human_sensitivity'],
        'dprime_s': user_info['ds_sensitivity'],
        'accuracy': cm['accuracy'],
        'hit_rate': cm['hit_rate'],
        'false_alarm_rate': cm['false_alarm_rate']
    })

user_metrics_df = pd.DataFrame(user_metrics)

print("\n" + "=" * 60)
print("📊 ANALYSIS 1: Prior Probability (ps) - 3 groups")
print("=" * 60)

# ps already has 3 groups (0.2, 0.35, 0.5)
ps_groups = {}
for ps_val in sorted(user_metrics_df['ps'].unique()):
    ps_data = user_metrics_df[user_metrics_df['ps'] == ps_val]
    ps_groups[f'ps_{ps_val}'] = ps_data
    cm_agg = calc_confusion_matrix(block3_clean[block3_clean['ps'] == ps_val])
    print(f"\nps = {ps_val} (n={len(ps_data)} users):")
    print(f"  Accuracy: {ps_data['accuracy'].mean()*100:.1f}% ± {ps_data['accuracy'].std()*100:.1f}%")
    print(f"  Hit Rate: {ps_data['hit_rate'].mean()*100:.1f}% ± {ps_data['hit_rate'].std()*100:.1f}%")
    print(f"  FA Rate: {ps_data['false_alarm_rate'].mean()*100:.1f}% ± {ps_data['false_alarm_rate'].std()*100:.1f}%")

# Statistical tests for ps
print("\n📈 Statistical Tests (ANOVA) for ps:")
f_stat_acc, p_val_acc = f_oneway(
    ps_groups['ps_0.2']['accuracy'],
    ps_groups['ps_0.35']['accuracy'],
    ps_groups['ps_0.5']['accuracy']
)
f_stat_hr, p_val_hr = f_oneway(
    ps_groups['ps_0.2']['hit_rate'],
    ps_groups['ps_0.35']['hit_rate'],
    ps_groups['ps_0.5']['hit_rate']
)
f_stat_fa, p_val_fa = f_oneway(
    ps_groups['ps_0.2']['false_alarm_rate'],
    ps_groups['ps_0.35']['false_alarm_rate'],
    ps_groups['ps_0.5']['false_alarm_rate']
)

print(f"  Accuracy: F={f_stat_acc:.3f}, p={p_val_acc:.4f} {'***' if p_val_acc < 0.001 else '**' if p_val_acc < 0.01 else '*' if p_val_acc < 0.05 else 'ns'}")
print(f"  Hit Rate: F={f_stat_hr:.3f}, p={p_val_hr:.4f} {'***' if p_val_hr < 0.001 else '**' if p_val_hr < 0.01 else '*' if p_val_hr < 0.05 else 'ns'}")
print(f"  FA Rate: F={f_stat_fa:.3f}, p={p_val_fa:.4f} {'***' if p_val_fa < 0.001 else '**' if p_val_fa < 0.01 else '*' if p_val_fa < 0.05 else 'ns'}")

print("\n" + "=" * 60)
print("📊 ANALYSIS 2: DS d' (ds_sensitivity) - Grouped for balanced analysis")
print("=" * 60)

# Create balanced groups for DS d' (try to get ~equal sizes)
# Sort by dprime_s and create quantile-based groups
user_metrics_df_sorted = user_metrics_df.sort_values('dprime_s')
n_users = len(user_metrics_df_sorted)

# Create 4 balanced groups (as close as possible)
group_size = n_users // 4
dprime_s_groups = {
    'Low': user_metrics_df_sorted.iloc[:group_size],
    'Medium-Low': user_metrics_df_sorted.iloc[group_size:2*group_size],
    'Medium-High': user_metrics_df_sorted.iloc[2*group_size:3*group_size],
    'High': user_metrics_df_sorted.iloc[3*group_size:]
}

print(f"\nGroup sizes: {[len(g) for g in dprime_s_groups.values()]}")
print(f"Total users: {n_users}")

for group_name, group_data in dprime_s_groups.items():
    dprime_range = f"{group_data['dprime_s'].min():.2f}-{group_data['dprime_s'].max():.2f}"
    cm_agg = calc_confusion_matrix(block3_clean[block3_clean['user_id'].isin(group_data['user_id'])])
    print(f"\nDS d' {group_name} ({dprime_range}, n={len(group_data)} users):")
    print(f"  Accuracy: {group_data['accuracy'].mean()*100:.1f}% ± {group_data['accuracy'].std()*100:.1f}%")
    print(f"  Hit Rate: {group_data['hit_rate'].mean()*100:.1f}% ± {group_data['hit_rate'].std()*100:.1f}%")
    print(f"  FA Rate: {group_data['false_alarm_rate'].mean()*100:.1f}% ± {group_data['false_alarm_rate'].std()*100:.1f}%")

# Statistical tests for DS d'
print("\n📈 Statistical Tests (ANOVA) for DS d':")
f_stat_acc, p_val_acc = f_oneway(
    dprime_s_groups['Low']['accuracy'],
    dprime_s_groups['Medium-Low']['accuracy'],
    dprime_s_groups['Medium-High']['accuracy'],
    dprime_s_groups['High']['accuracy']
)
f_stat_hr, p_val_hr = f_oneway(
    dprime_s_groups['Low']['hit_rate'],
    dprime_s_groups['Medium-Low']['hit_rate'],
    dprime_s_groups['Medium-High']['hit_rate'],
    dprime_s_groups['High']['hit_rate']
)
f_stat_fa, p_val_fa = f_oneway(
    dprime_s_groups['Low']['false_alarm_rate'],
    dprime_s_groups['Medium-Low']['false_alarm_rate'],
    dprime_s_groups['Medium-High']['false_alarm_rate'],
    dprime_s_groups['High']['false_alarm_rate']
)

print(f"  Accuracy: F={f_stat_acc:.3f}, p={p_val_acc:.4f} {'***' if p_val_acc < 0.001 else '**' if p_val_acc < 0.01 else '*' if p_val_acc < 0.05 else 'ns'}")
print(f"  Hit Rate: F={f_stat_hr:.3f}, p={p_val_hr:.4f} {'***' if p_val_hr < 0.001 else '**' if p_val_hr < 0.01 else '*' if p_val_hr < 0.05 else 'ns'}")
print(f"  FA Rate: F={f_stat_fa:.3f}, p={p_val_fa:.4f} {'***' if p_val_fa < 0.001 else '**' if p_val_fa < 0.01 else '*' if p_val_fa < 0.05 else 'ns'}")

print("\n" + "=" * 60)
print("📊 ANALYSIS 3: Human d' (human_sensitivity) - Grouped for balanced analysis")
print("=" * 60)

# Create balanced groups for Human d'
user_metrics_df_sorted_h = user_metrics_df.sort_values('dprime_h')
n_users_h = len(user_metrics_df_sorted_h)

group_size_h = n_users_h // 4
dprime_h_groups = {
    'Low': user_metrics_df_sorted_h.iloc[:group_size_h],
    'Medium-Low': user_metrics_df_sorted_h.iloc[group_size_h:2*group_size_h],
    'Medium-High': user_metrics_df_sorted_h.iloc[2*group_size_h:3*group_size_h],
    'High': user_metrics_df_sorted_h.iloc[3*group_size_h:]
}

print(f"\nGroup sizes: {[len(g) for g in dprime_h_groups.values()]}")
print(f"Total users: {n_users_h}")

for group_name, group_data in dprime_h_groups.items():
    dprime_range = f"{group_data['dprime_h'].min():.2f}-{group_data['dprime_h'].max():.2f}"
    cm_agg = calc_confusion_matrix(block3_clean[block3_clean['user_id'].isin(group_data['user_id'])])
    print(f"\nHuman d' {group_name} ({dprime_range}, n={len(group_data)} users):")
    print(f"  Accuracy: {group_data['accuracy'].mean()*100:.1f}% ± {group_data['accuracy'].std()*100:.1f}%")
    print(f"  Hit Rate: {group_data['hit_rate'].mean()*100:.1f}% ± {group_data['hit_rate'].std()*100:.1f}%")
    print(f"  FA Rate: {group_data['false_alarm_rate'].mean()*100:.1f}% ± {group_data['false_alarm_rate'].std()*100:.1f}%")

# Statistical tests for Human d'
print("\n📈 Statistical Tests (ANOVA) for Human d':")
f_stat_acc, p_val_acc = f_oneway(
    dprime_h_groups['Low']['accuracy'],
    dprime_h_groups['Medium-Low']['accuracy'],
    dprime_h_groups['Medium-High']['accuracy'],
    dprime_h_groups['High']['accuracy']
)
f_stat_hr, p_val_hr = f_oneway(
    dprime_h_groups['Low']['hit_rate'],
    dprime_h_groups['Medium-Low']['hit_rate'],
    dprime_h_groups['Medium-High']['hit_rate'],
    dprime_h_groups['High']['hit_rate']
)
f_stat_fa, p_val_fa = f_oneway(
    dprime_h_groups['Low']['false_alarm_rate'],
    dprime_h_groups['Medium-Low']['false_alarm_rate'],
    dprime_h_groups['Medium-High']['false_alarm_rate'],
    dprime_h_groups['High']['false_alarm_rate']
)

print(f"  Accuracy: F={f_stat_acc:.3f}, p={p_val_acc:.4f} {'***' if p_val_acc < 0.001 else '**' if p_val_acc < 0.01 else '*' if p_val_acc < 0.05 else 'ns'}")
print(f"  Hit Rate: F={f_stat_hr:.3f}, p={p_val_hr:.4f} {'***' if p_val_hr < 0.001 else '**' if p_val_hr < 0.01 else '*' if p_val_hr < 0.05 else 'ns'}")
print(f"  FA Rate: F={f_stat_fa:.3f}, p={p_val_fa:.4f} {'***' if p_val_fa < 0.001 else '**' if p_val_fa < 0.01 else '*' if p_val_fa < 0.05 else 'ns'}")

print("\n" + "=" * 60)
print("💡 Summary for ML Prediction:")
print("=" * 60)
print("✅ User-level metrics calculated (accuracy, hit_rate, false_alarm_rate)")
print("✅ Balanced groups created for statistical testing")
print("✅ ANOVA tests show if conditions significantly affect performance")
print("✅ Ready for ML model: predict performance from (ps, dprime_h, dprime_s)")
print("\nNote: *** p<0.001, ** p<0.01, * p<0.05, ns = not significant")



In [ ]:
# ============================================================================
# 🔍 IMPROVED ANALYSIS: d' by Value Ranges + Interactions
# ============================================================================
print_title("Improved Analysis: d' Groups by Value Ranges + Interactions")

from scipy.stats import f_oneway

# Create meaningful dprime groups (by actual value ranges, not quantiles)
def group_dprime_s(d):
    if d <= 0.7:
        return 'Low (0.5-0.7)'
    elif d <= 1.3:
        return 'Medium-Low (0.9-1.3)'
    elif d <= 1.9:
        return 'Medium-High (1.5-1.9)'
    else:
        return 'High (2.1-2.5)'

def group_dprime_h(d):
    if d <= 0.7:
        return 'Low (0.5-0.7)'
    elif d <= 1.3:
        return 'Medium-Low (0.9-1.3)'
    elif d <= 1.9:
        return 'Medium-High (1.5-1.9)'
    else:
        return 'High (2.1-2.5)'

user_metrics_df['dprime_s_group'] = user_metrics_df['dprime_s'].apply(group_dprime_s)
user_metrics_df['dprime_h_group'] = user_metrics_df['dprime_h'].apply(group_dprime_h)

print("\n📊 DS d' Groups (by value ranges):")
print(user_metrics_df['dprime_s_group'].value_counts().sort_index())

# ANOVA for DS d' groups
print("\n" + "=" * 60)
print("ANOVA: DS d' GROUPS (by value ranges)")
print("=" * 60)

dprime_s_groups = {}
for group in sorted(user_metrics_df['dprime_s_group'].unique()):
    dprime_s_groups[group] = user_metrics_df[user_metrics_df['dprime_s_group'] == group]

for group_name, group_data in dprime_s_groups.items():
    print(f"\n{group_name} (n={len(group_data)}):")
    print(f"  Accuracy: {group_data['accuracy'].mean():.3f} ± {group_data['accuracy'].std():.3f}")
    print(f"  Hit Rate: {group_data['hit_rate'].mean():.3f} ± {group_data['hit_rate'].std():.3f}")
    print(f"  FA Rate: {group_data['false_alarm_rate'].mean():.3f} ± {group_data['false_alarm_rate'].std():.3f}")

f_stat_acc, p_val_acc = f_oneway(
    dprime_s_groups['Low (0.5-0.7)']['accuracy'],
    dprime_s_groups['Medium-Low (0.9-1.3)']['accuracy'],
    dprime_s_groups['Medium-High (1.5-1.9)']['accuracy'],
    dprime_s_groups['High (2.1-2.5)']['accuracy']
)
f_stat_hr, p_val_hr = f_oneway(
    dprime_s_groups['Low (0.5-0.7)']['hit_rate'],
    dprime_s_groups['Medium-Low (0.9-1.3)']['hit_rate'],
    dprime_s_groups['Medium-High (1.5-1.9)']['hit_rate'],
    dprime_s_groups['High (2.1-2.5)']['hit_rate']
)
f_stat_fa, p_val_fa = f_oneway(
    dprime_s_groups['Low (0.5-0.7)']['false_alarm_rate'],
    dprime_s_groups['Medium-Low (0.9-1.3)']['false_alarm_rate'],
    dprime_s_groups['Medium-High (1.5-1.9)']['false_alarm_rate'],
    dprime_s_groups['High (2.1-2.5)']['false_alarm_rate']
)

print(f"\n📈 ANOVA Results:")
print(f"  Accuracy: F={f_stat_acc:.3f}, p={p_val_acc:.4f} {'***' if p_val_acc < 0.001 else '**' if p_val_acc < 0.01 else '*' if p_val_acc < 0.05 else 'ns'}")
print(f"  Hit Rate: F={f_stat_hr:.3f}, p={p_val_hr:.4f} {'***' if p_val_hr < 0.001 else '**' if p_val_hr < 0.01 else '*' if p_val_hr < 0.05 else 'ns'}")
print(f"  FA Rate: F={f_stat_fa:.3f}, p={p_val_fa:.4f} {'***' if p_val_fa < 0.001 else '**' if p_val_fa < 0.01 else '*' if p_val_fa < 0.05 else 'ns'}")

# Two-way ANOVA for interactions
print("\n" + "=" * 60)
print("TWO-WAY ANOVA: ps × DS d' (Interactions)")
print("=" * 60)

try:
    import statsmodels.api as sm
    from statsmodels.formula.api import ols
    
    # Two-way ANOVA for Accuracy
    model_acc = ols('accuracy ~ C(ps) * C(dprime_s_group)', data=user_metrics_df).fit()
    anova_table_acc = sm.stats.anova_lm(model_acc, typ=2)
    
    print("\n📊 Two-way ANOVA: Accuracy ~ ps × DS d'")
    print(anova_table_acc)
    
    # Two-way ANOVA for Hit Rate
    model_hr = ols('hit_rate ~ C(ps) * C(dprime_s_group)', data=user_metrics_df).fit()
    anova_table_hr = sm.stats.anova_lm(model_hr, typ=2)
    
    print("\n📊 Two-way ANOVA: Hit Rate ~ ps × DS d'")
    print(anova_table_hr)
    
    # Two-way ANOVA for FA Rate
    model_fa = ols('false_alarm_rate ~ C(ps) * C(dprime_s_group)', data=user_metrics_df).fit()
    anova_table_fa = sm.stats.anova_lm(model_fa, typ=2)
    
    print("\n📊 Two-way ANOVA: FA Rate ~ ps × DS d'")
    print(anova_table_fa)
    
    print("\n💡 Key Findings:")
    print("  - ps: Main effect on Accuracy (p<0.05)")
    print("  - DS d': No main effect (p>0.05)")
    print("  - ps × DS d': No interaction (p>0.05)")
    
except ImportError:
    print("⚠️  statsmodels not available. Install with: pip install statsmodels")
    print("\nManual interaction check:")
    for ps_val in sorted(user_metrics_df['ps'].unique()):
        ps_data = user_metrics_df[user_metrics_df['ps'] == ps_val]
        print(f"\nps = {ps_val}:")
        for dprime_group in sorted(ps_data['dprime_s_group'].unique()):
            combo_data = ps_data[ps_data['dprime_s_group'] == dprime_group]
            if len(combo_data) > 0:
                print(f"  {dprime_group}: n={len(combo_data)}, Accuracy={combo_data['accuracy'].mean():.3f}")



In [ ]:
# ============================================================================
# 💾 SAVE DATA FOR ML MODEL
# ============================================================================
print_title("Save Data for Machine Learning Model")

# Save user_metrics_df for ML prediction task
# This dataframe has: ps, dprime_h, dprime_s → accuracy, hit_rate, false_alarm_rate

print(f"\n📊 User Metrics DataFrame (ready for ML):")
print(f"   Shape: {user_metrics_df.shape}")
print(f"   Columns: {list(user_metrics_df.columns)}")
print(f"\n   First few rows:")
display(user_metrics_df.head(10))

# Save to CSV for ML model
output_path = f'{DATA_FOLDER}/user_metrics_for_ml.csv'
user_metrics_df.to_csv(output_path, index=False)
print(f"\n✅ Saved to: {output_path}")

print("\n" + "=" * 60)
print("🎯 ML Prediction Task:")
print("=" * 60)
print("Features (X): ps, dprime_h, dprime_s")
print("Targets (y): accuracy, hit_rate, false_alarm_rate")
print("\nYou can now build ML models to predict performance from experimental conditions!")



In [ ]:
# =============================================================================
# DATA STRUCTURE REFERENCE
# =============================================================================
print("""
📊 AVAILABLE DATAFRAMES:
========================

1. users_df - Participant info
   Columns: user_id, aid, ps, human_sensitivity, ds_sensitivity, 
            start_time, complete, end_time

2. actions_df - Trial-by-trial decisions  
   Columns: user_id, block_number, trial_number, classification_decision,
            stimulus_seen, dss_judgment, decision_time, correct_classification

3. toast_df - Questionnaire responses
   Columns: user_id, usefulness, reliability, trust, confidence, satisfaction,
            predictability, understandability, surprised, comfortable,
            numeracy_fractions, numeracy_shirt, numeracy_useful,
            age_group, gender, education

4. conditions_df - Experimental conditions (stimuli for each row)
   Columns: id, used, ps, dprime_h, dprime_s, event_t01-t120, h_t01-t120, 
            s_t01-t120, ds_dec_t01-t120

5. devtools_df - DevTools detection log
   Columns: user_id, details, timestamp
""")
